# Model 3: End-to-End PPG → BP → ECG Pipeline

This notebook combines both pre-trained models into a **unified end-to-end pipeline** that:
1. Takes a raw **PPG waveform** as input
2. Estimates **Blood Pressure** (SBP, DBP) via Model 1
3. Generates a full **12-lead ECG** conditioned on that BP via Model 2

It also performs **fine-tuning** of the combined system using VitalDB PPG data bridged through simulated BP to the Brugada-HUCA ECG distribution.

## Prerequisites
Run **Notebook 1** and **Notebook 2** first to generate:
- `best_ppg_to_bp.pth`
- `best_bp_to_ecg.pth`
- `bp_scaler.pkl`
- `bp_bounds.pkl`
- `ppg_bp_cache.pkl`

In [ ]:
! pip install vitaldb wfdb torch torchvision torchaudio scikit-learn matplotlib numpy pandas tqdm joblib

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import wfdb
from tqdm import tqdm
import warnings
import joblib
import pickle

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device: {device}')

## 1. Re-define Model Architectures (must match Notebooks 1 & 2)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1 ARCHITECTURE  –  PPG → BP
# ══════════════════════════════════════════════════════════════════════════════

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )
    def forward(self, x):
        w = self.fc(x.mean(dim=2))
        return x * w.unsqueeze(2)


class ResBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=pad), nn.BatchNorm1d(channels), nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size, padding=pad), nn.BatchNorm1d(channels),
        )
        self.attn = ChannelAttention(channels)
        self.act  = nn.GELU()
    def forward(self, x):
        return self.act(self.attn(self.conv(x)) + x)


class PPGtoBP(nn.Module):
    def __init__(self, win_len=625, lstm_hidden=128, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv1d(1,32,15,padding=7,stride=2), nn.BatchNorm1d(32), nn.GELU())
        self.layer1 = nn.Sequential(nn.Conv1d(32,64,3,stride=2,padding=1), ResBlock1D(64), ResBlock1D(64))
        self.layer2 = nn.Sequential(nn.Conv1d(64,128,3,stride=2,padding=1), ResBlock1D(128), ResBlock1D(128))
        self.layer3 = nn.Sequential(nn.Conv1d(128,256,3,stride=2,padding=1), ResBlock1D(256))
        self.bilstm = nn.LSTM(256, lstm_hidden, lstm_layers, batch_first=True, bidirectional=True, dropout=dropout)
        self.query  = nn.Linear(lstm_hidden*2, 1)
        self.regressor = nn.Sequential(
            nn.Linear(lstm_hidden*2,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Linear(64,2)
        )
    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        x = x.permute(0,2,1)
        x, _ = self.bilstm(x)
        w = torch.softmax(self.query(x), dim=1)
        x = (x * w).sum(dim=1)
        return self.regressor(x)


# ══════════════════════════════════════════════════════════════════════════════
# MODEL 2 ARCHITECTURE  –  BP → ECG (CVAE)
# ══════════════════════════════════════════════════════════════════════════════

LATENT_DIM = 64
BP_DIM     = 2
N_SAMPLES  = 1200
N_LEADS    = 12

class ConvBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=2, padding=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=padding),
            nn.BatchNorm1d(out_ch), nn.GELU()
        )
    def forward(self, x): return self.net(x)


class ConvTransBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=2, padding=3, out_padding=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose1d(in_ch, out_ch, kernel, stride=stride, padding=padding, output_padding=out_padding),
            nn.BatchNorm1d(out_ch), nn.GELU()
        )
    def forward(self, x): return self.net(x)


class CVAEEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, bp_dim=BP_DIM):
        super().__init__()
        self.bp_embed = nn.Linear(bp_dim, 4 * 1200)
        self.cnn = nn.Sequential(
            ConvBlock1D(12+4, 32, 15, 2, 7),
            ConvBlock1D(32,  64,  9, 2, 4),
            ConvBlock1D(64, 128,  7, 2, 3),
            ConvBlock1D(128,256,  5, 2, 2),
            ConvBlock1D(256,512,  5, 3, 2),
        )
        self.fc_mu      = nn.Linear(512*25, latent_dim)
        self.fc_log_var = nn.Linear(512*25, latent_dim)
    def forward(self, ecg, bp):
        bp_feat = self.bp_embed(bp).view(bp.size(0), 4, 1200)
        x = torch.cat([ecg, bp_feat], dim=1)
        x = self.cnn(x).view(x.size(0), -1)
        return self.fc_mu(x), self.fc_log_var(x)


class CVAEDecoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, bp_dim=BP_DIM):
        super().__init__()
        self.fc = nn.Linear(latent_dim + bp_dim, 512*25)
        self.deconv = nn.Sequential(
            ConvTransBlock1D(512,256, 5, 3, 2, 2),
            ConvTransBlock1D(256,128, 5, 2, 2, 1),
            ConvTransBlock1D(128, 64, 7, 2, 3, 1),
            ConvTransBlock1D( 64, 32, 9, 2, 4, 1),
            ConvTransBlock1D( 32, 12,15, 2, 7, 1),
        )
        self.out_act = nn.Tanh()
    def forward(self, z, bp):
        x = self.fc(torch.cat([z, bp], dim=1)).view(z.size(0), 512, 25)
        return self.out_act(self.deconv(x))


class CVAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, bp_dim=BP_DIM):
        super().__init__()
        self.encoder    = CVAEEncoder(latent_dim, bp_dim)
        self.decoder    = CVAEDecoder(latent_dim, bp_dim)
        self.latent_dim = latent_dim
    def reparameterize(self, mu, log_var):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * log_var)
    def forward(self, ecg, bp):
        mu, lv = self.encoder(ecg, bp)
        return self.decoder(self.reparameterize(mu, lv), bp), mu, lv
    def generate(self, bp, n_samples=1):
        z = torch.randn(n_samples, self.latent_dim, device=bp.device)
        if bp.shape[0] == 1 and n_samples > 1:
            bp = bp.expand(n_samples, -1)
        return self.decoder(z, bp)


print('Architectures defined.')

## 2. Load Pre-trained Weights

In [ ]:
# ─── Load Model 1 ─────────────────────────────────────────────────────────────
ppg_bp_model = PPGtoBP().to(device)
ppg_bp_model.load_state_dict(torch.load('best_ppg_to_bp.pth', map_location=device))
ppg_bp_model.eval()
print('Model 1 (PPG → BP) loaded.')

# ─── Load Model 2 ─────────────────────────────────────────────────────────────
bp_ecg_model = CVAE().to(device)
bp_ecg_model.load_state_dict(torch.load('best_bp_to_ecg.pth', map_location=device))
bp_ecg_model.eval()
print('Model 2 (BP → ECG) loaded.')

# ─── Load scalers / bounds ────────────────────────────────────────────────────
bp_scaler = joblib.load('bp_scaler.pkl')   # sklearn StandardScaler for Model 1 output
bp_bounds  = joblib.load('bp_bounds.pkl')  # {'sbp': (70,200), 'dbp': (40,130)}
print('Scalers loaded.')

## 3. End-to-End Pipeline Module

In [ ]:
class PPGtoECGPipeline(nn.Module):
    """
    Unified end-to-end model:
      PPG window (B,1,625) → [SBP_norm, DBP_norm] (B,2) → ECG (B,12,1200)

    The two sub-models share the same parameter spaces as in Notebooks 1 & 2.
    A lightweight BP adapter (2-layer MLP) converts Model-1 BP output format
    to Model-2 conditioning format without full re-scaling.
    """
    def __init__(self, ppg_bp_model, bp_ecg_model, bp_scaler, bp_bounds):
        super().__init__()
        self.ppg_bp  = ppg_bp_model
        self.bp_ecg  = bp_ecg_model

        # Register scaler stats as buffers (non-trainable, move with .to(device))
        mean_ = torch.tensor(bp_scaler.mean_,   dtype=torch.float32)
        scale_ = torch.tensor(bp_scaler.scale_, dtype=torch.float32)
        self.register_buffer('bp_mean',  mean_)
        self.register_buffer('bp_scale', scale_)

        # bp_bounds for Model-2 normalization
        self.sbp_lo, self.sbp_hi = bp_bounds['sbp']
        self.dbp_lo, self.dbp_hi = bp_bounds['dbp']

        # Learnable BP adapter (fine-tuning bridge)
        self.bp_adapter = nn.Sequential(
            nn.Linear(2, 16), nn.GELU(),
            nn.Linear(16, 2), nn.Tanh()   # output in [-1, 1] for CVAE conditioning
        )

    def bp_model1_to_model2(self, bp_norm):
        """
        Convert Model-1 output (StandardScaler-normalized) → mmHg → Model-2 range.
        """
        # Inverse-standardize to mmHg
        bp_mmhg = bp_norm * self.bp_scale + self.bp_mean  # (B,2)
        sbp_mmhg = bp_mmhg[:, 0]
        dbp_mmhg = bp_mmhg[:, 1]

        # Normalize to [-1, 1] for CVAE
        sbp_n = (sbp_mmhg - self.sbp_lo) / (self.sbp_hi - self.sbp_lo) * 2 - 1
        dbp_n = (dbp_mmhg - self.dbp_lo) / (self.dbp_hi - self.dbp_lo) * 2 - 1

        return torch.stack([sbp_n, dbp_n], dim=1)  # (B,2)

    def forward(self, ppg, use_adapter=True):
        """
        Full pipeline forward pass.
        ppg: (B, 1, 625)
        Returns: ecg (B,12,1200), bp_mmhg (B,2)
        """
        # Stage 1: PPG → BP
        bp_norm1 = self.ppg_bp(ppg)                  # (B,2) standardized

        # Inverse-scale to mmHg for logging
        bp_mmhg = bp_norm1 * self.bp_scale + self.bp_mean

        # Convert to Model 2 conditioning format
        bp_cond = self.bp_model1_to_model2(bp_norm1)  # (B,2) in [-1,1]

        # Optional learnable adapter for fine-tuning
        if use_adapter:
            bp_cond = self.bp_adapter(bp_cond)

        # Stage 2: BP → ECG (generation)
        z = torch.randn(ppg.size(0), self.bp_ecg.latent_dim, device=ppg.device)
        ecg = self.bp_ecg.decoder(z, bp_cond)         # (B,12,1200)

        return ecg, bp_mmhg

    def get_bp(self, ppg):
        """Get BP estimate only (runs Model 1 only)."""
        with torch.no_grad():
            bp_norm1 = self.ppg_bp(ppg)
        return (bp_norm1 * self.bp_scale + self.bp_mean).cpu().numpy()


pipeline = PPGtoECGPipeline(ppg_bp_model, bp_ecg_model, bp_scaler, bp_bounds).to(device)
total_params = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
print(f'Total pipeline parameters: {total_params:,}')

## 4. Load Data for Fine-tuning

In [ ]:
# ─── Load PPG cache from Notebook 1 ──────────────────────────────────────────
CACHE_FILE = 'ppg_bp_cache.pkl'
if not os.path.exists(CACHE_FILE):
    raise FileNotFoundError('Run Notebook 1 first to generate ppg_bp_cache.pkl')

with open(CACHE_FILE, 'rb') as f:
    ppg_windows, sbp_labels, dbp_labels = pickle.load(f)

FS_PPG   = 125
WIN_SAMP = 625

# Normalize PPG
mu  = ppg_windows.mean(axis=1, keepdims=True)
std = ppg_windows.std(axis=1,  keepdims=True) + 1e-8
ppg_norm = ((ppg_windows - mu) / std).astype(np.float32)

# Normalize BP
bp_arr  = np.stack([sbp_labels, dbp_labels], axis=1).astype(np.float32)
bp_std_norm = bp_scaler.transform(bp_arr).astype(np.float32)

print(f'PPG windows: {ppg_norm.shape}')
print(f'BP labels  : {bp_arr.shape}')

In [ ]:
# ─── Load Brugada-HUCA ECGs ───────────────────────────────────────────────────
DATASET_ROOT = r'D:\BPI-Net\brugada-huca-12-lead-ecg-recordings-for-the-study-of-brugada-syndrome-1.0.0\brugada-huca-12-lead-ecg-recordings-for-the-study-of-brugada-syndrome-1.0.0'
DATA_DIR     = os.path.join(DATASET_ROOT, 'files')
META_PATH    = os.path.join(DATASET_ROOT, 'metadata.csv')

def load_ecg_data(data_dir, meta_path):
    metadata = pd.read_csv(meta_path)
    ecg_list = []
    for _, row in tqdm(metadata.iterrows(), total=len(metadata)):
        pid = str(int(row['patient_id']))
        rec_path = os.path.join(data_dir, pid, pid)
        try:
            record = wfdb.rdrecord(rec_path)
            sig = record.p_signal
            if sig is None or sig.shape != (1200, 12): continue
            if np.any(np.isnan(sig)) or np.any(np.isinf(sig)): continue
            # Normalize per-lead
            s = sig.T.astype(np.float32)  # (12, 1200)
            s = (s - s.mean(axis=1, keepdims=True)) / (s.std(axis=1, keepdims=True) + 1e-8)
            ecg_list.append(s)
        except: continue
    return np.array(ecg_list)

ecg_data = load_ecg_data(DATA_DIR, META_PATH)
print(f'ECG data shape: {ecg_data.shape}')  # (363, 12, 1200)

In [ ]:
class CombinedDataset(Dataset):
    """
    Pairs PPG windows (with their BP) with randomly sampled real ECGs.
    The loss encourages the generated ECG to match the ECG distribution.
    """
    def __init__(self, ppg, bp_std, ecg_pool):
        self.ppg      = torch.tensor(ppg[:, np.newaxis, :], dtype=torch.float32)
        self.bp_std   = torch.tensor(bp_std, dtype=torch.float32)
        self.ecg_pool = torch.tensor(ecg_pool, dtype=torch.float32)  # random reference pool

    def __len__(self):
        return len(self.ppg)

    def __getitem__(self, idx):
        # Pair with a random real ECG as a distribution target
        ecg_idx = np.random.randint(len(self.ecg_pool))
        return self.ppg[idx], self.bp_std[idx], self.ecg_pool[ecg_idx]


dataset  = CombinedDataset(ppg_norm, bp_std_norm, ecg_data)
n_total  = len(dataset)
n_train  = int(0.80 * n_total)
n_val    = n_total - n_train

train_set, val_set = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {n_train} | Val: {n_val}')

## 5. Fine-tuning Strategy

In [ ]:
"""
Fine-tuning strategy:
  - Freeze Model 1 (PPG→BP) backbone — keep pretrained BP estimation stable
  - Freeze Model 2 encoder (we only generate, not reconstruct)
  - Train: BP adapter + Model 2 decoder (with small LR)

Loss = Distribution matching MSE (generated ECG vs real ECG pool)
     + BP consistency loss (predicted BP should stay close to original)
"""

# ─── Freeze PPG→BP and CVAE encoder ──────────────────────────────────────────
for param in pipeline.ppg_bp.parameters():
    param.requires_grad = False

for param in pipeline.bp_ecg.encoder.parameters():
    param.requires_grad = False

# Trainable: adapter + decoder
trainable = [
    {'params': pipeline.bp_adapter.parameters(),         'lr': 1e-3},
    {'params': pipeline.bp_ecg.decoder.parameters(),     'lr': 5e-5},
]

optimizer  = optim.Adam(trainable)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

trainable_params = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
print(f'Trainable params for fine-tuning: {trainable_params:,}')

In [ ]:
def combined_loss(gen_ecg, ref_ecg, bp_pred_norm, bp_label_norm, w_ecg=1.0, w_bp=0.5):
    """
    gen_ecg      : (B,12,1200) generated ECG
    ref_ecg      : (B,12,1200) real reference ECG (from Brugada-HUCA pool)
    bp_pred_norm : (B,2) BP predicted by Model 1 (for consistency check)
    bp_label_norm: (B,2) true BP normalized labels
    """
    ecg_loss = F.mse_loss(gen_ecg, ref_ecg)
    bp_loss  = F.mse_loss(bp_pred_norm, bp_label_norm)
    return w_ecg * ecg_loss + w_bp * bp_loss, ecg_loss.item(), bp_loss.item()


EPOCHS    = 30
PATIENCE  = 8

train_losses, val_losses = [], []
best_val = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────────
    pipeline.train()
    # Keep frozen parts in eval mode
    pipeline.ppg_bp.eval()
    pipeline.bp_ecg.encoder.eval()

    tr_loss = 0
    for ppg, bp_std_lbl, ref_ecg in tqdm(train_loader, desc=f'Epoch {epoch:2d} Train', leave=False):
        ppg, bp_std_lbl, ref_ecg = ppg.to(device), bp_std_lbl.to(device), ref_ecg.to(device)

        gen_ecg, _ = pipeline(ppg, use_adapter=True)
        with torch.no_grad():
            bp_pred_raw = pipeline.ppg_bp(ppg)  # frozen prediction

        loss, el, bl = combined_loss(gen_ecg, ref_ecg, bp_pred_raw, bp_std_lbl)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(pipeline.parameters(), 1.0)
        optimizer.step()
        tr_loss += loss.item() * ppg.size(0)

    tr_loss /= len(train_loader.dataset)

    # ── Validation ─────────────────────────────────────────────────────────────
    pipeline.eval()
    vl_loss = 0
    with torch.no_grad():
        for ppg, bp_std_lbl, ref_ecg in val_loader:
            ppg, bp_std_lbl, ref_ecg = ppg.to(device), bp_std_lbl.to(device), ref_ecg.to(device)
            gen_ecg, _ = pipeline(ppg, use_adapter=True)
            bp_pred_raw = pipeline.ppg_bp(ppg)
            loss, _, _ = combined_loss(gen_ecg, ref_ecg, bp_pred_raw, bp_std_lbl)
            vl_loss += loss.item() * ppg.size(0)

    vl_loss /= len(val_loader.dataset)
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)

    if vl_loss < best_val:
        best_val = vl_loss
        torch.save(pipeline.state_dict(), 'best_ppg_to_ecg_pipeline.pth')
        patience_counter = 0
    else:
        patience_counter += 1

    print(f'Epoch {epoch:2d} | Train: {tr_loss:.4f} | Val: {vl_loss:.4f}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'Best val loss: {best_val:.4f}')

## 6. Full Pipeline Evaluation & Visualization

In [ ]:
# ─── Loss curves ──────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Combined Loss')
plt.title('Fine-tuning Curves – Full Pipeline')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ─── Load best pipeline ────────────────────────────────────────────────────────
pipeline.load_state_dict(torch.load('best_ppg_to_ecg_pipeline.pth', map_location=device))
pipeline.eval()

# ─── End-to-End demo ──────────────────────────────────────────────────────────
sample_ppg = ppg_norm[0]   # (625,)
ppg_t = torch.tensor(sample_ppg[np.newaxis, np.newaxis, :], dtype=torch.float32).to(device)

with torch.no_grad():
    gen_ecg, bp_pred = pipeline(ppg_t, use_adapter=True)

gen_ecg_np = gen_ecg.squeeze().cpu().numpy()   # (12, 1200)
sbp_pred   = bp_pred[0, 0].item()
dbp_pred   = bp_pred[0, 1].item()

print(f'Predicted BP: {sbp_pred:.1f}/{dbp_pred:.1f} mmHg')
print(f'True BP:      {sbp_labels[0]:.1f}/{dbp_labels[0]:.1f} mmHg')

In [ ]:
# ─── Plot full pipeline output ────────────────────────────────────────────────
LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
t_ppg = np.arange(WIN_SAMP) / FS_PPG
t_ecg = np.arange(1200) / 100

fig = plt.figure(figsize=(18, 14))
fig.suptitle(f'Full Pipeline: PPG → BP ({sbp_pred:.0f}/{dbp_pred:.0f} mmHg) → ECG', fontsize=14)

# PPG input
ax_ppg = fig.add_subplot(5, 3, (1, 3))
ax_ppg.plot(t_ppg, sample_ppg, color='green', linewidth=1)
ax_ppg.set_title('Input PPG Signal')
ax_ppg.set_xlabel('Time (s)')
ax_ppg.set_ylabel('Amplitude (norm)')
ax_ppg.grid(True, alpha=0.3)
ax_ppg.text(0.02, 0.95,
    f'→ Predicted BP: {sbp_pred:.0f}/{dbp_pred:.0f} mmHg  |  True: {sbp_labels[0]:.0f}/{dbp_labels[0]:.0f} mmHg',
    transform=ax_ppg.transAxes, fontsize=10, va='top',
    bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

# Generated ECG leads
for i, lead in enumerate(LEAD_NAMES):
    ax = fig.add_subplot(5, 3, i + 4)
    ax.plot(t_ecg, gen_ecg_np[i], linewidth=0.8, color='steelblue')
    ax.set_title(f'Generated ECG – {lead}')
    ax.set_ylabel('mV')
    ax.grid(True, linestyle=':', alpha=0.4)

for ax in fig.axes[-3:]:
    ax.set_xlabel('Time (s)')

plt.tight_layout()
plt.show()

In [ ]:
# ─── Batch evaluation ─────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error

pipeline.eval()
all_bp_pred, all_bp_true = [], []

with torch.no_grad():
    for ppg, bp_std_lbl, _ in val_loader:
        ppg = ppg.to(device)
        _, bp_pred = pipeline(ppg, use_adapter=True)
        all_bp_pred.append(bp_pred.cpu().numpy())
        # Inverse-standardize true BP
        bp_true_mmhg = bp_scaler.inverse_transform(bp_std_lbl.numpy())
        all_bp_true.append(bp_true_mmhg)

bp_pred_arr = np.vstack(all_bp_pred)
bp_true_arr = np.vstack(all_bp_true)

for i, name in enumerate(['SBP', 'DBP']):
    mae  = mean_absolute_error(bp_true_arr[:, i], bp_pred_arr[:, i])
    rmse = np.sqrt(np.mean((bp_true_arr[:, i] - bp_pred_arr[:, i])**2))
    print(f'{name}: MAE={mae:.2f} mmHg | RMSE={rmse:.2f} mmHg')

In [ ]:
# ─── Convenience inference function ──────────────────────────────────────────
def ppg_to_ecg(ppg_window_raw: np.ndarray) -> dict:
    """
    Full pipeline inference from a raw PPG window.
    Args:
        ppg_window_raw: 1D numpy array of 625 PPG samples at 125 Hz
    Returns:
        dict with 'SBP', 'DBP' (mmHg) and 'ECG' (12×1200 array)
    """
    # Normalize
    x = (ppg_window_raw - ppg_window_raw.mean()) / (ppg_window_raw.std() + 1e-8)
    x_t = torch.tensor(x[np.newaxis, np.newaxis, :], dtype=torch.float32).to(device)

    pipeline.eval()
    with torch.no_grad():
        gen_ecg, bp = pipeline(x_t, use_adapter=True)

    return {
        'SBP': round(bp[0, 0].item(), 1),
        'DBP': round(bp[0, 1].item(), 1),
        'ECG': gen_ecg.squeeze().cpu().numpy(),  # (12, 1200)
    }


# Test it
result = ppg_to_ecg(ppg_windows[10])
print(f"BP estimate: {result['SBP']}/{result['DBP']} mmHg")
print(f"ECG shape:   {result['ECG'].shape}")
print('\nPipeline model saved as: best_ppg_to_ecg_pipeline.pth')

## Summary

| Stage | Input | Output | Architecture |
|-------|-------|--------|--------------|
| **Model 1** | PPG (625 samples) | SBP, DBP (mmHg) | CNN-BiLSTM + Attention |
| **Model 2** | BP (2 values) | 12-lead ECG (1200 samples) | Conditional VAE |
| **Pipeline** | PPG → BP → ECG | Full end-to-end | Combined + fine-tuned adapter |

Saved artifacts:
- `best_ppg_to_bp.pth` – Model 1 weights
- `best_bp_to_ecg.pth` – Model 2 weights
- `best_ppg_to_ecg_pipeline.pth` – Full pipeline weights
- `bp_scaler.pkl` – BP StandardScaler
- `bp_bounds.pkl` – BP normalization bounds